# MoP + DivPO — Phase 3 Training Only (Kaggle, Maximum GPU)

**Prerequisites:**
- SFT adapters already exist at `DasonTio/mop-divpo-coauthor/sft/{persona}/`.
- DivPO preference datasets already exist at `DasonTio/mop-divpo-divpo-data/{persona}.jsonl`.

## Kaggle settings (before running)

| Setting | Value |
|---|---|
| Accelerator | **T4 x2** (Settings → Accelerator → GPU T4 x2) |
| Internet | On |
| Secret | `HF_TOKEN` = your HuggingFace write token |

## How to run in background (Save Version)

1. Click **Save Version** (top-right) → **Save & Run All (Commit)** → **Save**
2. Close browser / sleep laptop — job continues on Kaggle servers
3. Kaggle emails when done. Check output at **Your Work → Notebooks**

## Session plan

| Run | Phase | Est. time |
|---|---|---|
| Save Version 1 | Check prepared DivPO data + train all personas | **~1.5–2h total** |

## GPU optimizations active

| Component | Optimization |
|---|---|
| Training | Accelerate DDP (both T4s), fused AdamW, gradient checkpointing, SDPA, group_by_length |

---
## Cell 1 — Install dependencies

**Save Version:** no restart needed (fresh kernel).
**Interactive:** restart kernel once after this cell, then continue.

In [ ]:
import os
INTERACTIVE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Interactive"

!pip install -q --upgrade transformers peft trl accelerate bitsandbytes datasets huggingface_hub sentence-transformers
!pip uninstall -y -q torchao

if INTERACTIVE:
    print("\n>>> INTERACTIVE MODE: restart kernel now, then continue from Cell 2. <<<")
else:
    print("Save Version mode — continuing.")

---
## Cell 2 — Setup

Credentials + repo + working directory + accelerate config for DDP training.
**Run at the start of every session.**

In [ ]:
import os
import sys
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

REPO_DIR = "/kaggle/working/mop-divpo-llm-counter-argument"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/DasonTio/mop-divpo-llm-counter-argument.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --quiet

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Write accelerate config for 2-GPU DDP training (Phase 3)
accel_cfg = """\
compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
downcast_bf16: 'no'
gpu_ids: all
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
use_cpu: false
"""
with open("/tmp/accel_config.yaml", "w") as f:
    f.write(accel_cfg)

import torch
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
if n_gpu == 0:
    print("NO GPU — go to Notebook settings → Accelerator → T4 x2")
print(f"CWD : {os.getcwd()}")
print(f"HF  : {os.environ['HF_TOKEN'][:8]}...")
print(f"Accelerate config: /tmp/accel_config.yaml ({n_gpu} GPUs)")
print("Ready.")

---
## Prepared DivPO Dataset Check

This notebook is training-only. It does not regenerate DivPO pairs.

If the four prepared JSONL files are already on the Hugging Face Hub, this cell only verifies them and training continues. If any file is missing, the notebook stops before spending GPU time.

In [ ]:
# Verify prepared DivPO preference data on HF Hub. No dataset generation happens here.
from huggingface_hub import list_repo_files
import os

DIVPO_DATA_REPO = "DasonTio/mop-divpo-divpo-data"
PERSONAS = ["contrarian", "systems_thinker", "cross_domain_analogist", "minimalist"]
expected = {f"{persona}.jsonl" for persona in PERSONAS}
files = set(list_repo_files(DIVPO_DATA_REPO, repo_type="dataset", token=os.environ["HF_TOKEN"]))
missing = sorted(expected - files)

print("DivPO data on Hub:")
for f in sorted(expected & files):
    print(" ", f)

if missing:
    raise RuntimeError(
        "Missing prepared DivPO data: " + ", ".join(missing) + "\n"
        "Run scripts/prepare_divpo_datasets.py separately before this training notebook."
    )

print("\nAll prepared DivPO datasets found — skipping dataset generation and continuing to training.")

---
## DivPO Training

Uses `accelerate launch` for true 2-GPU DDP training:
- Process 0 on `cuda:0`, Process 1 on `cuda:1`
- Each process: trainable model + ref model on its own GPU (no cross-device issues)
- Effective batch = 1 per device × 2 GPUs × 32 grad_accum = 64
- SDPA attention, fused AdamW, gradient checkpointing, fp16, max_length=384

**Conservative T4-safe settings. If this fits comfortably, batch size can be tuned upward later.**

In [ ]:
# DivPO training — all personas sequentially (auto-push per adapter)
import subprocess

PERSONAS = ["contrarian", "systems_thinker", "cross_domain_analogist", "minimalist"]
for persona in PERSONAS:
    print(f"\n=== DivPO training: {persona} ===\n")
    cmd = [
        "accelerate",
        "launch",
        "--config_file",
        "/tmp/accel_config.yaml",
        "--num_processes",
        "2",
        "--num_machines",
        "1",
        "--mixed_precision",
        "fp16",
        "--dynamo_backend",
        "no",
        "scripts/train_divpo.py",
        "--persona",
        persona,
        "--batch-size",
        "1",
        "--grad-accum",
        "32",
        "--max-length",
        "384",
    ]
    subprocess.run(cmd, check=True)


---
## Verify — Load DivPO adapter and generate

In [ ]:
import os
import warnings
import torch
from pathlib import Path
from huggingface_hub import list_repo_files
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REPO = "DasonTio/mop-divpo-coauthor"
PERSONAS = ["contrarian", "systems_thinker", "cross_domain_analogist", "minimalist"]
PERSONA_TO_VERIFY = None  # set to a persona string to force one; None auto-selects an available adapter
warnings.filterwarnings("ignore", message=r"Already found a `peft_config` attribute in the model\.", category=UserWarning)
warnings.filterwarnings("ignore", message=r"You are trying to modify a model with PEFT for a second time\.", category=UserWarning)

token = os.environ.get("HF_TOKEN") or None
files = set(list_repo_files(MODEL_REPO, repo_type="model", token=token))
local_root = Path("outputs/adapters/divpo")
available_local_personas = [
    persona for persona in PERSONAS
    if (local_root / persona / "adapter_config.json").exists()
]
available_hub_personas = [
    persona for persona in PERSONAS
    if f"divpo/{persona}/adapter_config.json" in files
]

if PERSONA_TO_VERIFY is not None:
    if PERSONA_TO_VERIFY not in PERSONAS:
        raise ValueError(f"Unknown PERSONA_TO_VERIFY={PERSONA_TO_VERIFY!r}; choose one of {PERSONAS}")
    resolved_persona = PERSONA_TO_VERIFY
else:
    candidates = available_local_personas or available_hub_personas
    if not candidates:
        available = sorted(f for f in files if f.startswith("divpo/"))
        raise RuntimeError(
            "No local or Hub DivPO adapters found.\n"
            f"Local checked: {local_root}\n"
            f"Available Hub divpo files: {available if available else 'none'}\n"
            "Run at least one training cell successfully, or check that push_adapter completed."
        )
    resolved_persona = candidates[0]
    print(f"Auto-selected DivPO persona for verification: {resolved_persona}")

local_adapter_dir = local_root / resolved_persona
hub_subfolder = f"divpo/{resolved_persona}"

if (local_adapter_dir / "adapter_config.json").exists():
    adapter_source = str(local_adapter_dir)
    adapter_kwargs = {}
    print(f"Loading local DivPO adapter: {adapter_source}")
else:
    expected_config = f"{hub_subfolder}/adapter_config.json"
    if expected_config not in files:
        available = sorted(f for f in files if f.startswith("divpo/"))
        raise RuntimeError(
            f"No local adapter found at {local_adapter_dir}, and Hub file {expected_config} is missing.\n"
            f"Available Hub divpo files: {available if available else 'none'}\n"
            "Run the training cell for this persona successfully, or check that push_adapter completed."
        )
    adapter_source = MODEL_REPO
    adapter_kwargs = {"subfolder": hub_subfolder, "token": token}
    print(f"Loading Hub DivPO adapter: {MODEL_REPO}/{hub_subfolder}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map="auto")
sft_subfolder = f"sft/{resolved_persona}"
model = PeftModel.from_pretrained(base, MODEL_REPO, subfolder=sft_subfolder, token=token)
model = PeftModel.from_pretrained(model, adapter_source, **adapter_kwargs)
model.eval()

prompt = "Generate a counter-argument to this claim:\n\nRemote work is strictly better for productivity."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.9, do_sample=True)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

# Free notebook-kernel GPU memory before running validation in a separate process.
del model, base, inputs, out
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## Pre-Evaluation Inference Validation

Runs a small persona-aware generation gate before full evaluation. Outputs are saved as JSONL for manual inspection.

In [ ]:
!python scripts/validate_divpo_inference.py --all --output outputs/evaluation/pre_eval_validation.jsonl